# 02 — Perfil Histórico e Risco de Não Recorrência | Consignado

## Objetivo

Este notebook aprofunda a análise das lojas que produziram Consignado até o **13º dia útil do mês anterior** e ainda não produziram até o mesmo dia útil do mês atual.

O objetivo não é assumir que todas essas lojas representam uma perda definitiva. Vamos investigar:

- **quando cada loja costuma começar a produzir Consignado no mês;**
- recorrência nos últimos 6–12 meses;
- frequência de produção;
- potencial histórico;
- ruptura do comportamento habitual;
- comportamento em outros produtos;
- diferença entre lojas `PERDEU` e `MANTEVE`;
- prioridade inicial de atuação.

A pergunta central é:

> **A ausência de Consignado até o 13º DU é normal para essa loja ou representa uma ruptura do seu padrão histórico?**

### Fonte principal
`TESTE..PROD_DIA_UTIL_BE`

### Premissa
Cada linha representa a produção de uma `CHAVE_LOJA` em um `PERIODO` e `QTD_DIA_UTIL_MES`.


In [ ]:
# 1. Bibliotecas e parâmetros

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from db import read_sql

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PERIODO_ANTERIOR = 202607
PERIODO_ATUAL = 202608
DU_CORTE = 13

# Histórico: ajuste se desejar trabalhar com 12 meses.
PERIODO_INICIO_HIST = 202602
PERIODO_FIM_HIST = PERIODO_ANTERIOR

PRODUTOS = {
    "CONSIG": ("QTD_CONSIG", "QTD_CONSIG_ACUM"),
    "CONTAS": ("QTD_CONTAS", "QTD_CONTAS_ACUM"),
    "SEGUROS": ("SEG_TOTAL", "SEG_TOTAL_ACUM"),
    "CRED_TOTAL": ("CRED_TOTAL", "CRED_TOTAL_ACUM"),
    "LIME": ("LIME", "LIME_ACUM"),
}

print(
    f"Coorte: {PERIODO_ANTERIOR} x {PERIODO_ATUAL} até {DU_CORTE}º DU | "
    f"Histórico: {PERIODO_INICIO_HIST} a {PERIODO_FIM_HIST}"
)


## 2. Extração

Para este notebook precisamos de duas visões:

1. **Coorte comparável:** mês anterior × mês atual até o 13º DU.
2. **Histórico:** meses anteriores completos, necessário para entender o padrão normal de cada loja.

A consulta abaixo busca tudo de uma vez. A conexão SQL fica em `db.py`.


In [ ]:
# 2. Extração (conexão em db.py)

query = f"""
SELECT
    *
FROM TESTE..PROD_DIA_UTIL_BE
WHERE PERIODO BETWEEN {PERIODO_INICIO_HIST} AND {PERIODO_ATUAL}
"""

print(query)

df = read_sql(query)

# Alternativa local (sem SQL):
# df = pd.read_csv("PROD_DIA_UTIL_BE.csv")


## 3. Auditoria

Antes de construir qualquer feature, confirme que existe no máximo uma linha por:

`CHAVE_LOJA + PERIODO + QTD_DIA_UTIL_MES`


In [ ]:
def auditar_base(df):
    chaves = ["CHAVE_LOJA", "PERIODO", "QTD_DIA_UTIL_MES"]

    print("Linhas:", f"{len(df):,}")
    print("Lojas:", f"{df['CHAVE_LOJA'].nunique():,}")
    print("Períodos:", sorted(df["PERIODO"].dropna().unique()))

    duplicadas = df.duplicated(chaves, keep=False)
    print("Linhas duplicadas:", int(duplicadas.sum()))

    resumo = (
        df.groupby("PERIODO")
          .agg(
              LOJAS=("CHAVE_LOJA", "nunique"),
              DU_MIN=("QTD_DIA_UTIL_MES", "min"),
              DU_MAX=("QTD_DIA_UTIL_MES", "max"),
              LINHAS=("CHAVE_LOJA", "size")
          )
    )

    display(resumo)

# auditar_base(df)


## 4. Reconstrução da coorte

O universo principal será formado pelas lojas que **já tinham produzido Consignado até o 13º DU do mês anterior**.

Dentro dele:

- `MANTEVE`: produziu novamente até o 13º DU atual.
- `PERDEU`: ainda não produziu até o 13º DU atual.

Também identificamos `ENTROU` para análises complementares.


In [ ]:
def valor_acumulado_ate_du(df, periodo, coluna_acum, du):
    base = df[
        (df["PERIODO"] == periodo) &
        (df["QTD_DIA_UTIL_MES"] <= du)
    ].copy()

    # Usa a última observação disponível até o DU.
    base = (
        base.sort_values(["CHAVE_LOJA", "QTD_DIA_UTIL_MES"])
            .groupby("CHAVE_LOJA", as_index=False)
            .tail(1)
    )

    return base[["CHAVE_LOJA", coluna_acum]].rename(
        columns={coluna_acum: f"{coluna_acum}_{periodo}"}
    )


def construir_coorte(df):
    ant = valor_acumulado_ate_du(
        df, PERIODO_ANTERIOR, "QTD_CONSIG_ACUM", DU_CORTE
    )
    atual = valor_acumulado_ate_du(
        df, PERIODO_ATUAL, "QTD_CONSIG_ACUM", DU_CORTE
    )

    base = ant.merge(atual, on="CHAVE_LOJA", how="outer").fillna(0)

    c_ant = f"QTD_CONSIG_ACUM_{PERIODO_ANTERIOR}"
    c_atual = f"QTD_CONSIG_ACUM_{PERIODO_ATUAL}"

    base["PROD_ANT"] = (base[c_ant] > 0).astype(int)
    base["PROD_ATUAL"] = (base[c_atual] > 0).astype(int)

    base["STATUS_CONSIG"] = np.select(
        [
            (base["PROD_ANT"] == 1) & (base["PROD_ATUAL"] == 1),
            (base["PROD_ANT"] == 1) & (base["PROD_ATUAL"] == 0),
            (base["PROD_ANT"] == 0) & (base["PROD_ATUAL"] == 1),
        ],
        ["MANTEVE", "PERDEU", "ENTROU"],
        default="INATIVA"
    )

    return base

# coorte = construir_coorte(df)
# display(coorte["STATUS_CONSIG"].value_counts().to_frame("LOJAS"))


In [ ]:
# CHECKPOINT DAS 502

# perdas = coorte.query("STATUS_CONSIG == 'PERDEU'").copy()
# mantidas = coorte.query("STATUS_CONSIG == 'MANTEVE'").copy()

# print("PERDEU :", len(perdas))
# print("MANTEVE:", len(mantidas))

# if len(perdas) != 502:
#     print(
#         "ATENÇÃO: o total encontrado é diferente de 502. "
#         "Revise granularidade, disponibilidade do 13º DU e regra da coorte."
#     )


# 5. Quando cada loja costuma começar a produzir?

Esta é a principal evolução deste notebook.

Para cada `CHAVE_LOJA + PERIODO`, calculamos o **primeiro dia útil com QTD_CONSIG > 0**.

Depois, no histórico da loja, calculamos:

- primeiro DU médio;
- primeiro DU mediano;
- primeiro DU mínimo/máximo;
- percentual de meses em que começou até o 5º, 10º e 13º DU;
- percentual de meses em que começou depois do 13º DU.

Isso ajuda a separar **atraso esperado** de **ruptura comportamental**.


In [ ]:
def primeiro_du_consignado(df_hist):
    prod = df_hist[df_hist["QTD_CONSIG"].fillna(0) > 0].copy()

    primeiro = (
        prod.groupby(["CHAVE_LOJA", "PERIODO"], as_index=False)
            .agg(
                PRIMEIRO_DU_CONSIG=("QTD_DIA_UTIL_MES", "min"),
                DIAS_COM_CONSIG=("QTD_DIA_UTIL_MES", "nunique"),
                QTD_CONSIG_MES=("QTD_CONSIG", "sum"),
                VLR_CONSIG_MES=("VLR_CONSIG", "sum")
            )
    )

    return primeiro


def features_primeiro_du(primeiro):
    def pct_ate(s, du):
        return (s <= du).mean() * 100

    feat = (
        primeiro.groupby("CHAVE_LOJA")
        .agg(
            MESES_COM_CONSIG=("PERIODO", "nunique"),
            PRIMEIRO_DU_MEDIO=("PRIMEIRO_DU_CONSIG", "mean"),
            PRIMEIRO_DU_MEDIANO=("PRIMEIRO_DU_CONSIG", "median"),
            PRIMEIRO_DU_MIN=("PRIMEIRO_DU_CONSIG", "min"),
            PRIMEIRO_DU_MAX=("PRIMEIRO_DU_CONSIG", "max"),
            DIAS_CONSIG_MEDIO=("DIAS_COM_CONSIG", "mean"),
            QTD_CONSIG_MEDIA_MES=("QTD_CONSIG_MES", "mean"),
            VLR_CONSIG_MEDIA_MES=("VLR_CONSIG_MES", "mean"),
            VLR_CONSIG_MEDIANA_MES=("VLR_CONSIG_MES", "median"),
        )
        .reset_index()
    )

    percentuais = (
        primeiro.groupby("CHAVE_LOJA")["PRIMEIRO_DU_CONSIG"]
        .agg(
            PCT_INICIO_ATE_DU5=lambda s: pct_ate(s, 5),
            PCT_INICIO_ATE_DU10=lambda s: pct_ate(s, 10),
            PCT_INICIO_ATE_DU13=lambda s: pct_ate(s, 13),
            PCT_INICIO_APOS_DU13=lambda s: (s > 13).mean() * 100
        )
        .reset_index()
    )

    return feat.merge(percentuais, on="CHAVE_LOJA", how="left")

# df_hist = df[
#     (df["PERIODO"] >= PERIODO_INICIO_HIST) &
#     (df["PERIODO"] <= PERIODO_FIM_HIST)
# ].copy()
#
# primeiro = primeiro_du_consignado(df_hist)
# feat_du = features_primeiro_du(primeiro)
#
# coorte = coorte.merge(feat_du, on="CHAVE_LOJA", how="left")


## 6. Recorrência histórica

Uma loja que produziu em **6/6 meses** e agora está zerada é muito diferente de uma que produziu em apenas **1/6**.

Criamos:

- número de meses observados;
- meses com Consignado;
- taxa de recorrência;
- classe de recorrência;
- produção média/mediana;
- quantidade média de operações;
- ticket histórico aproximado.


In [ ]:
def features_recorrencia(df_hist):
    meses_observados = sorted(df_hist["PERIODO"].dropna().unique())

    mensal = (
        df_hist.groupby(["CHAVE_LOJA", "PERIODO"], as_index=False)
        .agg(
            QTD_CONSIG_MES=("QTD_CONSIG", "sum"),
            VLR_CONSIG_MES=("VLR_CONSIG", "sum")
        )
    )

    # Garante meses sem produção para lojas presentes no histórico.
    lojas = df_hist["CHAVE_LOJA"].drop_duplicates()
    grade = pd.MultiIndex.from_product(
        [lojas, meses_observados],
        names=["CHAVE_LOJA", "PERIODO"]
    ).to_frame(index=False)

    mensal = grade.merge(
        mensal,
        on=["CHAVE_LOJA", "PERIODO"],
        how="left"
    ).fillna(0)

    mensal["PRODUZIU_CONSIG"] = (mensal["QTD_CONSIG_MES"] > 0).astype(int)

    feat = (
        mensal.groupby("CHAVE_LOJA")
        .agg(
            MESES_OBSERVADOS=("PERIODO", "nunique"),
            MESES_PROD_CONSIG=("PRODUZIU_CONSIG", "sum"),
            QTD_CONSIG_MEDIA_6M=("QTD_CONSIG_MES", "mean"),
            QTD_CONSIG_MEDIANA_6M=("QTD_CONSIG_MES", "median"),
            VLR_CONSIG_MEDIA_6M=("VLR_CONSIG_MES", "mean"),
            VLR_CONSIG_MEDIANA_6M=("VLR_CONSIG_MES", "median")
        )
        .reset_index()
    )

    feat["RECORRENCIA_6M"] = (
        feat["MESES_PROD_CONSIG"] / feat["MESES_OBSERVADOS"]
    )

    feat["CLASSE_RECORRENCIA"] = pd.cut(
        feat["RECORRENCIA_6M"],
        bins=[-0.01, 0.33, 0.66, 0.99, 1.01],
        labels=[
            "ESPORÁDICA",
            "INTERMITENTE",
            "RECORRENTE",
            "MUITO RECORRENTE"
        ]
    )

    feat["TICKET_HIST_APROX"] = np.where(
        feat["QTD_CONSIG_MEDIA_6M"] > 0,
        feat["VLR_CONSIG_MEDIA_6M"] / feat["QTD_CONSIG_MEDIA_6M"],
        np.nan
    )

    return feat, mensal

# feat_rec, mensal_consig = features_recorrencia(df_hist)
# coorte = coorte.merge(feat_rec, on="CHAVE_LOJA", how="left")


## 7. Índice de anormalidade da ausência atual

Não é um modelo preditivo. É uma regra analítica inicial para priorizar investigação.

Uma ausência até o 13º DU é mais anormal quando:

- a loja produz Consignado em muitos meses;
- normalmente começa antes do 13º DU;
- possui valor histórico relevante;
- possui vários dias de produção.

Criamos um score simples de 0–100, cujos pesos podem ser recalibrados após observar os dados.


In [ ]:
def normalizar_0_100(s):
    s = pd.to_numeric(s, errors="coerce")
    minimo, maximo = s.min(), s.max()

    if pd.isna(minimo) or pd.isna(maximo) or maximo == minimo:
        return pd.Series(0, index=s.index, dtype=float)

    return ((s - minimo) / (maximo - minimo) * 100).clip(0, 100)


def adicionar_score_anormalidade(base):
    out = base.copy()

    out["SCORE_RECORRENCIA"] = (
        out["RECORRENCIA_6M"].fillna(0) * 100
    ).clip(0, 100)

    out["SCORE_INICIO_CEDO"] = (
        out["PCT_INICIO_ATE_DU13"].fillna(0)
    ).clip(0, 100)

    out["SCORE_POTENCIAL"] = normalizar_0_100(
        np.log1p(out["VLR_CONSIG_MEDIA_6M"].fillna(0))
    )

    out["SCORE_FREQUENCIA"] = normalizar_0_100(
        out["DIAS_CONSIG_MEDIO"].fillna(0)
    )

    out["SCORE_ANORMALIDADE"] = (
        0.35 * out["SCORE_RECORRENCIA"] +
        0.35 * out["SCORE_INICIO_CEDO"] +
        0.20 * out["SCORE_POTENCIAL"] +
        0.10 * out["SCORE_FREQUENCIA"]
    ).round(1)

    return out

# coorte = adicionar_score_anormalidade(coorte)
# perdas = coorte.query("STATUS_CONSIG == 'PERDEU'").copy()


## 8. Classificação de risco comportamental

A classificação abaixo é intencionalmente interpretável:

- **RUPTURA FORTE:** recorrente e normalmente já teria produzido até o DU atual.
- **RUPTURA MODERADA:** há histórico relevante, mas o padrão é menos consistente.
- **POSSÍVEL ATRASO NORMAL:** costuma iniciar depois do corte.
- **PRODUÇÃO EVENTUAL:** pouca recorrência histórica.


In [ ]:
def classificar_risco_historico(row):
    rec = row.get("RECORRENCIA_6M", np.nan)
    pct13 = row.get("PCT_INICIO_ATE_DU13", np.nan)

    if pd.isna(rec):
        return "SEM HISTÓRICO"

    if rec >= 0.67 and pct13 >= 75:
        return "RUPTURA FORTE"

    if rec >= 0.50 and pct13 >= 50:
        return "RUPTURA MODERADA"

    if pct13 < 50 and rec >= 0.50:
        return "POSSÍVEL ATRASO NORMAL"

    if rec < 0.50:
        return "PRODUÇÃO EVENTUAL"

    return "MONITORAR"

# perdas["RISCO_HISTORICO"] = perdas.apply(
#     classificar_risco_historico, axis=1
# )
#
# display(
#     perdas["RISCO_HISTORICO"]
#     .value_counts()
#     .to_frame("LOJAS")
# )


# 9. PERDEU × MANTEVE

Este é um dos blocos mais importantes.

Não queremos apenas descrever as perdas. Queremos descobrir **o que diferencia quem perdeu de quem continuou produzindo**.

A comparação inicial considera:

- recorrência;
- primeiro DU;
- frequência;
- valor histórico;
- ticket.


In [ ]:
def resumo_perdeu_manteve(coorte):
    universo = coorte[
        coorte["STATUS_CONSIG"].isin(["PERDEU", "MANTEVE"])
    ].copy()

    metricas = [
        "RECORRENCIA_6M",
        "PRIMEIRO_DU_MEDIANO",
        "PCT_INICIO_ATE_DU13",
        "DIAS_CONSIG_MEDIO",
        "VLR_CONSIG_MEDIA_6M",
        "TICKET_HIST_APROX",
    ]

    metricas = [m for m in metricas if m in universo.columns]

    return (
        universo.groupby("STATUS_CONSIG")[metricas]
        .median()
        .T
        .reset_index()
        .rename(columns={"index": "METRICA"})
    )

# comparativo = resumo_perdeu_manteve(coorte)
# display(comparativo)


## 10. Distribuição do primeiro dia de produção

Esse gráfico mostra **quando as lojas historicamente entram em Consignado**.

A linha vertical no 13º DU ajuda a visualizar quanto da população normalmente já teria produzido até o ponto atual.


In [ ]:
def plot_primeiro_du(primeiro, du_corte=13):
    dados = primeiro["PRIMEIRO_DU_CONSIG"].dropna()

    fig, ax = plt.subplots(figsize=(11, 5))
    bins = np.arange(dados.min() - 0.5, dados.max() + 1.5, 1)

    ax.hist(dados, bins=bins)
    ax.axvline(du_corte, linestyle="--", linewidth=1.5)

    ax.set_title(
        "Quando as lojas costumam iniciar produção de Consignado?",
        loc="left",
        fontweight="bold"
    )
    ax.set_xlabel("Primeiro dia útil com produção")
    ax.set_ylabel("Observações loja-mês")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

# plot_primeiro_du(primeiro, DU_CORTE)


## 11. Curva histórica de ativação

A curva abaixo responde:

> **Até cada dia útil, qual percentual das lojas produtoras já costuma ter iniciado Consignado?**

Ela é particularmente útil para avaliar se o 13º DU é cedo ou tarde para considerar uma ausência como sinal de risco.


In [ ]:
def curva_ativacao_historica(primeiro):
    max_du = int(primeiro["PRIMEIRO_DU_CONSIG"].max())
    total = len(primeiro)

    linhas = []
    for du in range(1, max_du + 1):
        pct = (primeiro["PRIMEIRO_DU_CONSIG"] <= du).mean() * 100
        linhas.append({"DIA_UTIL": du, "PCT_ATIVADO": pct})

    return pd.DataFrame(linhas)


def plot_curva_ativacao(curva, du_corte=13):
    fig, ax = plt.subplots(figsize=(11, 5))

    ax.plot(
        curva["DIA_UTIL"],
        curva["PCT_ATIVADO"],
        marker="o",
        markersize=4
    )
    ax.axvline(du_corte, linestyle="--", linewidth=1.5)

    ponto = curva.loc[curva["DIA_UTIL"] == du_corte]
    if not ponto.empty:
        pct = ponto["PCT_ATIVADO"].iloc[0]
        ax.annotate(
            f"{pct:.1f}% até o {du_corte}º DU",
            (du_corte, pct),
            xytext=(12, -25),
            textcoords="offset points"
        )

    ax.set_title(
        "Curva histórica de ativação do Consignado",
        loc="left",
        fontweight="bold"
    )
    ax.set_xlabel("Dia útil")
    ax.set_ylabel("% acumulado das ativações")
    ax.set_ylim(0, 105)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

# curva = curva_ativacao_historica(primeiro)
# plot_curva_ativacao(curva, DU_CORTE)


# 12. Outros produtos no mesmo estágio do mês

Agora cruzamos a ruptura histórica com o comportamento atual de:

- Contas;
- Seguros;
- Crédito Total;
- LIME.

A comparação deve ser **13º DU × 13º DU**.

Isso permite diferenciar:

- perda específica de Consignado;
- possível mudança de mix;
- deterioração comercial ampla.


In [ ]:
def snapshot_produtos(df, periodo, du):
    base = df[
        (df["PERIODO"] == periodo) &
        (df["QTD_DIA_UTIL_MES"] <= du)
    ].copy()

    base = (
        base.sort_values(["CHAVE_LOJA", "QTD_DIA_UTIL_MES"])
            .groupby("CHAVE_LOJA", as_index=False)
            .tail(1)
    )

    colunas = ["CHAVE_LOJA"]
    for _, acum in PRODUTOS.values():
        if acum in base.columns:
            colunas.append(acum)

    return base[colunas]


def adicionar_comparacao_produtos(coorte, df):
    ant = snapshot_produtos(df, PERIODO_ANTERIOR, DU_CORTE)
    atual = snapshot_produtos(df, PERIODO_ATUAL, DU_CORTE)

    ant = ant.rename(columns={
        c: f"{c}_ANT" for c in ant.columns if c != "CHAVE_LOJA"
    })
    atual = atual.rename(columns={
        c: f"{c}_ATUAL" for c in atual.columns if c != "CHAVE_LOJA"
    })

    out = (
        coorte.merge(ant, on="CHAVE_LOJA", how="left")
              .merge(atual, on="CHAVE_LOJA", how="left")
    )

    for produto, (_, acum) in PRODUTOS.items():
        ca = f"{acum}_ANT"
        cb = f"{acum}_ATUAL"

        if ca not in out.columns or cb not in out.columns:
            continue

        out[ca] = out[ca].fillna(0)
        out[cb] = out[cb].fillna(0)

        out[f"{produto}_DELTA"] = out[cb] - out[ca]
        out[f"{produto}_VAR_PCT"] = np.where(
            out[ca] != 0,
            (out[cb] / out[ca] - 1) * 100,
            np.nan
        )

    return out

# coorte = adicionar_comparacao_produtos(coorte, df)
# perdas = coorte.query("STATUS_CONSIG == 'PERDEU'").copy()


## 13. Hipótese comportamental

Estas categorias **não são causas comprovadas**. São hipóteses orientadas por evidência para priorizar investigação comercial.


In [ ]:
def movimento(var_pct, tolerancia=10):
    if pd.isna(var_pct):
        return "SEM BASE"
    if var_pct > tolerancia:
        return "CRESCE"
    if var_pct < -tolerancia:
        return "CAI"
    return "ESTÁVEL"


def classificar_fenomeno(row):
    cred = movimento(row.get("CRED_TOTAL_VAR_PCT"))
    lime = movimento(row.get("LIME_VAR_PCT"))
    contas = movimento(row.get("CONTAS_VAR_PCT"))
    seguros = movimento(row.get("SEGUROS_VAR_PCT"))

    outros = [cred, lime, contas, seguros]
    n_cai = sum(x == "CAI" for x in outros)
    n_preserva = sum(x in ["CRESCE", "ESTÁVEL"] for x in outros)

    if lime == "CRESCE" and cred in ["CRESCE", "ESTÁVEL"]:
        return "POSSÍVEL MUDANÇA DE MIX"

    if n_cai >= 3:
        return "DETERIORAÇÃO AMPLA"

    if n_preserva >= 3:
        return "PERDA ESPECÍFICA DE CONSIGNADO"

    return "COMPORTAMENTO MISTO"

# perdas["FENOMENO"] = perdas.apply(classificar_fenomeno, axis=1)
# display(perdas["FENOMENO"].value_counts().to_frame("LOJAS"))


# 14. Priorização inicial

A prioridade combina duas ideias:

### Ruptura
A ausência atual é incomum para a loja?

### Potencial
A loja historicamente representa uma oportunidade relevante?

O score abaixo é exploratório. Ele não deve ser tratado como decisão definitiva até que os pesos sejam validados com o negócio.


In [ ]:
def prioridade_inicial(perdas):
    out = perdas.copy()

    if "SCORE_ANORMALIDADE" not in out.columns:
        return out

    potencial = normalizar_0_100(
        np.log1p(out["VLR_CONSIG_MEDIA_6M"].fillna(0))
    )

    out["SCORE_PRIORIDADE"] = (
        0.65 * out["SCORE_ANORMALIDADE"] +
        0.35 * potencial
    ).round(1)

    out["PRIORIDADE"] = pd.cut(
        out["SCORE_PRIORIDADE"],
        bins=[-1, 40, 60, 80, 101],
        labels=["BAIXA", "MÉDIA", "ALTA", "MUITO ALTA"]
    )

    return out

# perdas = prioridade_inicial(perdas)
#
# display(
#     perdas["PRIORIDADE"]
#     .value_counts()
#     .to_frame("LOJAS")
# )


## 15. Ranking para investigação

O ranking final deve servir como **fila de investigação**, e não como prova de causalidade.

A recomendação é começar pelas lojas:

1. com ruptura forte;
2. historicamente recorrentes;
3. que normalmente produzem antes do 13º DU;
4. com alto potencial histórico;
5. que continuam ativas em outros produtos.


In [ ]:
# colunas_ranking = [
#     "CHAVE_LOJA",
#     "RISCO_HISTORICO",
#     "FENOMENO",
#     "RECORRENCIA_6M",
#     "PRIMEIRO_DU_MEDIANO",
#     "PCT_INICIO_ATE_DU13",
#     "VLR_CONSIG_MEDIA_6M",
#     "TICKET_HIST_APROX",
#     "CRED_TOTAL_VAR_PCT",
#     "LIME_VAR_PCT",
#     "CONTAS_VAR_PCT",
#     "SEGUROS_VAR_PCT",
#     "SCORE_ANORMALIDADE",
#     "SCORE_PRIORIDADE",
#     "PRIORIDADE"
# ]
#
# colunas_ranking = [c for c in colunas_ranking if c in perdas.columns]
#
# ranking = (
#     perdas[colunas_ranking]
#     .sort_values("SCORE_PRIORIDADE", ascending=False)
# )
#
# display(ranking.head(30))


# 16. Insights automáticos iniciais

Este bloco gera frases quantitativas para ajudar a leitura da análise depois que os dados estiverem carregados.


In [ ]:
def gerar_insights(perdas):
    total = len(perdas)

    if total == 0:
        print("Nenhuma perda encontrada.")
        return

    print(f"Total de lojas sem recorrência até o {DU_CORTE}º DU: {total:,}")

    if "RISCO_HISTORICO" in perdas:
        ruptura = perdas["RISCO_HISTORICO"].eq("RUPTURA FORTE").sum()
        print(
            f"Ruptura forte: {ruptura:,} lojas "
            f"({ruptura / total * 100:.1f}%)."
        )

    if "PCT_INICIO_ATE_DU13" in perdas:
        normalmente_ate = (perdas["PCT_INICIO_ATE_DU13"] >= 75).sum()
        print(
            f"{normalmente_ate:,} lojas ({normalmente_ate / total * 100:.1f}%) "
            f"historicamente iniciam até o {DU_CORTE}º DU em pelo menos 75% "
            "dos meses em que produzem."
        )

    if "FENOMENO" in perdas:
        print("\nHipóteses comportamentais:")
        dist = perdas["FENOMENO"].value_counts()
        for nome, qtd in dist.items():
            print(f"- {nome}: {qtd:,} ({qtd / total * 100:.1f}%)")

# gerar_insights(perdas)


# 17. Exportação

O arquivo de saída será a ponte para o próximo estudo territorial.

Sugestão de próximo notebook:

**`03_territorio_e_benchmark.ipynb`**

Ele deverá incorporar:

- UF e município;
- porte populacional;
- capital/interior;
- Gerência de Gestão;
- GC III;
- GC;
- taxa de não recorrência;
- benchmark entre lojas semelhantes;
- efeito praça × efeito carteira × efeito loja.


In [ ]:
# Path("outputs").mkdir(exist_ok=True)

# perdas.to_excel(
#     "outputs/02_perfil_historico_perdas_consignado.xlsx",
#     index=False
# )

# ranking.to_excel(
#     "outputs/02_ranking_prioridade_consignado.xlsx",
#     index=False
# )

print("Notebook 02 estruturado e pronto para execução.")
